<a href="https://colab.research.google.com/github/wangdian412/-/blob/master/clang_m7_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Make sure that you have set up API KEYS in Secrets on the left menu icon.  For upstash_redis, you need to create account and database.
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
%pip install -q langchain openai langchain-openai faiss-gpu langchain-community upstash_redis

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from langchain_community.chat_message_histories import UpstashRedisChatMessageHistory, RedisChatMessageHistory

Please change URL and TOKEN from your registration.

In [ ]:
UPSTASH_URL="https://usw1-enough-goat-33313.upstash.io"
UPSTASH_TOKEN="AYIhACQgMGY0NWUyYTAtODdiMy00YTZiLWE5ZTgtYjlkNDMxMDBmZDUwZWM5MzFkMzVlYzA3NDM3NmFmODVlZTNlYTVlNTJmMTM="

In [ ]:
history = UpstashRedisChatMessageHistory(
    url=UPSTASH_URL,
    token=UPSTASH_TOKEN,
    session_id="jl-chat1"
)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a friendly AI assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])

In [ ]:
model = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.6,
    openai_api_key=openai_api_key,
)

In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    chat_memory=history,
)

chain = prompt | model

In [ ]:
chain = LLMChain(
    llm=model,
    prompt=prompt,
    memory=memory,
    verbose=True
)

Prompt 1

In [ ]:
msg1 = { "input": "My name is Indiana Jones." }
resp1 = chain.invoke(msg1)
print(resp1['text'])

Now go to Data Browser in Upstash console. You find that the above conversation is stored in the database.

Prompt 2

In [ ]:
msg2 = { "input": "What is my name?" }
resp2 = chain.invoke(msg2)
print(resp2['text'])